# Feature Engineering — Traffic Collision Severity Classification

This notebook constructs the final feature set used to train binary classifiers that predict whether a traffic collision results in a **fatal** or **non-fatal** outcome.

The pipeline is organized into five stages:

| Stage | Description |
|---|---|
| 1. Target Encoding | Convert string class labels to binary integers |
| 2. Temporal Features | Decompose the accident date into meaningful time components |
| 3. Domain Knowledge Features | Construct risk indicators grounded in traffic safety research |
| 4. Interaction Features | Capture compounded risk from combinations of conditions |
| 5. Categorical Encoding | One-hot encode remaining categorical columns for model compatibility |

**Input:** `cleaned_data.csv` — 91,843 records, 11 features  
**Output:** `final_features.csv` — 91,843 records, 60 features

In [100]:
import pandas as pd
pd.set_option('display.max_columns', 100)  # ensure all columns are visible in output

import matplotlib.pyplot as plt
import seaborn as sns

In [101]:
# Load the cleaned dataset
df = pd.read_csv('cleaned_data.csv')

print(f"Dataset shape: {df.shape[0]:,} records and {df.shape[1]} features.")
df.head(3)

Dataset shape: 91,843 records and 11 features.


,Classification_Of_Accident,Accident_Date,Initial_Impact_Type,Road_1_Surface_Condition,Environment_Condition_1,Light,Traffic_Control,num_of_vehicles,num_of_pedestrians,num_of_bicycles,num_of_motorcycles
0,02 - Non-fatal injury,2017-01-04,Angle,Loose snow,Snow,Daylight,No control,2,0,0,0
1,02 - Non-fatal injury,2017-01-05,Turning movement,Wet,Clear,Dark,Traffic signal,2,0,0,0
2,02 - Non-fatal injury,2017-01-23,SMV other,Dry,Clear,Dark,Traffic signal,1,1,0,0


---
## Step 1 — Target Variable Encoding

The target column `Classification_Of_Accident` contains string labels. We map them to binary integers so classifiers can process them:

- `1` → Fatal injury *(minority class — 170 records)*
- `0` → Non-fatal injury *(majority class — 91,673 records)*

In [102]:
# Map string class labels to binary integers
df['Classification_Of_Accident'] = df['Classification_Of_Accident'].map({
    '01 - Fatal injury': 1,
    '02 - Non-fatal injury': 0
})

print("Class distribution after encoding:")
print(df['Classification_Of_Accident'].value_counts())

Class distribution after encoding:
Classification_Of_Accident
0    91673
1      170
Name: count, dtype: int64


---
## Step 2 — Temporal Feature Extraction

The `Accident_Date` column contains the full collision date. Rather than using it as-is, we break it into four features that capture meaningful time-based patterns in accident severity:

| Feature | Description |
|---|---|
| `month` | Calendar month (1–12) — captures seasonal trends |
| `day_of_week` | Day of the week (0=Mon … 6=Sun) |
| `is_weekend` | Binary flag: 1 if Saturday or Sunday, 0 otherwise |
| `season` | Season grouping — 0=Winter, 1=Spring, 2=Summer, 3=Fall |

The original date column is dropped after extraction since all of its information is captured in the new features.

In [103]:
df['Accident_Date'] = pd.to_datetime(df['Accident_Date'])

# Extract individual time components from the date
df['month'] = df['Accident_Date'].dt.month        
df['day_of_week'] = df['Accident_Date'].dt.dayofweek # 0 = Monday 
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)  

# Map each month to one of four seasons
df['season'] = df['month'].map({
    12: 0, 1: 0, 2: 0,   # Winter
    3: 1,  4: 1, 5: 1,   # Spring
    6: 2,  7: 2, 8: 2,   # Summer
    9: 3, 10: 3, 11: 3   # Fall
})

# Drop the original date column 
df.drop(columns=['Accident_Date'], inplace=True)

print("New dataset shape:", df.shape)

New dataset shape: (91843, 14)


---
## Step 3 — Domain Knowledge Features

These features are constructed using established traffic safety knowledge. Each one transforms a raw attribute into a direct signal for fatality risk, grouped across four risk dimensions:

### 3a — Vulnerable Road User Involvement

Pedestrians, cyclists, and motorcyclists lack the structural protection of an enclosed vehicle, making them significantly more likely to sustain fatal injuries in a collision.

We create two features:
- `vulnerable_users` — total count of vulnerable road users involved
- `has_vulnerable_user` — binary flag (1 if any vulnerable user was present)

In [104]:
# Total count of vulnerable road users involved (pedestrians + cyclists + motorcyclists)
df['vulnerable_users'] = (
    df['num_of_pedestrians'] + df['num_of_bicycles'] + df['num_of_motorcycles']
)

# Binary flag: 1 if at least one vulnerable road user was present, 0 otherwise
df['has_vulnerable_user'] = (df['vulnerable_users'] > 0).astype(int)

print("Vulnerable user involvement rate:", f"{df['has_vulnerable_user'].mean():.2%}")

Vulnerable user involvement rate: 4.53%


In [105]:
# the new vulnerable road users features
df[['Classification_Of_Accident','num_of_pedestrians',
    'num_of_bicycles','num_of_motorcycles','vulnerable_users','has_vulnerable_user']].head(3)

,Classification_Of_Accident,num_of_pedestrians,num_of_bicycles,num_of_motorcycles,vulnerable_users,has_vulnerable_user
0,0,0,0,0,0,0
1,0,0,0,0,0,0
2,0,1,0,0,1,1


In [106]:
df['Light'].value_counts()

Light
Daylight    61824
Dark        20754
Dusk         4201
Unknown      2707
Dawn         2305
Other          38
Name: count, dtype: int64

### 3b — Darkness Exposure

Collisions occurring in dark conditions are associated with reduced driver reaction time and lower pedestrian/cyclist visibility. We create a binary feature `is_dark` from the `Light` column to flag these incidents.

In [107]:
# to handle any case-insensitive make all values lowercase
df['is_dark'] = df['Light'].str.lower().str.contains('dark', na=False).astype(int)

print("Dark condition rate:", f"{df['is_dark'].mean():.2%}")

Dark condition rate: 22.60%


### 3c — Road Surface and Environmental Hazards

Adverse road surface and weather conditions reduce traction, extend stopping distances, and impair driver control. These are all factors that increase the likelihood of a severe outcome.

- `hazardous_surface` — flags surfaces known to reduce grip (snow, ice, wet, slush)
- `poor_environment` — flags atmospheric conditions that reduce visibility or traction (snow, rain, fog, freezing rain)

> **String matching note:** The label-stripping step in data cleaning (`str.split('-').str.get(1)`) leaves a **leading space** on each value (e.g., `" Snow"` not `"Snow"`). We call `.str.strip()` before `.isin()` to remove it. Values in the match list must also **exactly match** the cleaned data including capitalisation (e.g., `"Freezing Rain"` not `"Freezing rain"`) and the full label string (e.g., `"Fog, mist, smoke, dust"` not `"Fog"`).

In [108]:
# Road surfaces that reduce vehicle traction and increase stopping distance
hazardous_surfaces = ['Loose snow', 'Packed snow', 'Wet', 'Slush', 'Ice']

# .str.strip() removes the leading space produced by the label-stripping step in data cleaning
# Without it, " Wet" != "Wet" and would lead to incorrect counts
df['hazardous_surface'] = df['Road_1_Surface_Condition'].str.strip().isin(hazardous_surfaces).astype(int)

# Values must exactly match the cleaned data — capitalisation and full string matter
# e.g., "Freezing Rain" (not "Freezing rain"), "Fog, mist, smoke, dust" (not "Fog")
poor_conditions = ['Snow', 'Rain', 'Freezing Rain', 'Fog, mist, smoke, dust', 'Drifting Snow']
df['poor_environment'] = df['Environment_Condition_1'].str.strip().isin(poor_conditions).astype(int)

print("Hazardous surface rate:", f"{df['hazardous_surface'].mean():.2%}")
print("Poor environment rate: ", f"{df['poor_environment'].mean():.2%}")

Hazardous surface rate: 31.94%
Poor environment rate:  20.76%


### 3d — Traffic Control and Vehicle Involvement

Locations with no traffic control allow vehicles to travel at unregulated speeds, increasing both conflict probability and impact severity. Multi-vehicle collisions introduce additional kinetic energy and complexity compared to single-vehicle incidents.

- `no_control` — binary flag for collisions at uncontrolled locations
- `multi_vehicle` — binary flag for collisions involving more than one vehicle

> **String matching note:** Same leading-space issue applies here. `.str.strip()` is applied before the equality check so that `" No control"` correctly matches `"No control"`.

In [109]:
# Flag collisions at locations with no traffic control mechanism (e.g., no signal, no stop sign)
# .str.strip() removes the leading space from label cleaning (e.g., " No control" -> "No control")
df['no_control'] = (df['Traffic_Control'].str.strip() == 'No control').astype(int)

# Flag multi-vehicle collisions (more than one vehicle involved)
df['multi_vehicle'] = (df['num_of_vehicles'] > 1).astype(int)

print("No control rate:   ", f"{df['no_control'].mean():.2%}")
print("Multi-vehicle rate:", f"{df['multi_vehicle'].mean():.2%}")

No control rate:    46.48%
Multi-vehicle rate: 77.91%


---
## Step 4 — Interaction Features

Interaction features capture scenarios where multiple risk factors occur simultaneously, producing a compounded risk that is greater than either factor alone.

- `vulnerable_x_dark` — product of `has_vulnerable_user` and `is_dark`: flags collisions where a vulnerable road user is present under dark conditions, combining reduced driver visibility with the physical vulnerability of the road user

In [110]:
# Interaction: vulnerable road user present AND dark lighting conditions
# Multiplying two binary flags produces 1 only when both conditions are true simultaneously
df['vulnerable_x_dark'] = df['has_vulnerable_user'] * df['is_dark']

print("Vulnerable user + dark rate:", f"{df['vulnerable_x_dark'].mean():.2%}")
df[['Classification_Of_Accident','has_vulnerable_user','is_dark','vulnerable_x_dark']].sample(4)

Vulnerable user + dark rate: 1.04%


,Classification_Of_Accident,has_vulnerable_user,is_dark,vulnerable_x_dark
31258,0,0,1,0
70453,0,0,0,0
87340,0,0,0,0
79337,0,0,0,0


---
## Step 5 — Categorical Encoding (One-Hot Encoding)

The five remaining categorical columns are converted to binary dummy variables using **one-hot encoding**.

`drop_first=True` removes one dummy per column to avoid multicollinearity (the dropped category becomes the implicit reference group).

| Column | Categories | Dummies Created |
|---|---|---|
| `Initial_Impact_Type` | 8 | 7 |
| `Road_1_Surface_Condition` | 11 | 10 |
| `Environment_Condition_1` | 9 | 8 |
| `Light` | 6 | 5 |
| `Traffic_Control` | 14 | 13 |

This expands the feature space from 22 columns to **60 total features**.

In [111]:
# Columns to be one-hot encoded
categorical_cols = [
    'Initial_Impact_Type',
    'Road_1_Surface_Condition',
    'Environment_Condition_1',
    'Light',
    'Traffic_Control'
]

# Apply one-hot encoding — drop_first removes one dummy per column to avoid multicollinearity
# Cast all values to int to keep the dataset compact (booleans -> 0/1)
df_f = pd.get_dummies(df, columns=categorical_cols, drop_first=True).astype(int)

df_f.sample(2)

,Classification_Of_Accident,num_of_vehicles,num_of_pedestrians,num_of_bicycles,num_of_motorcycles,month,day_of_week,is_weekend,season,vulnerable_users,has_vulnerable_user,is_dark,hazardous_surface,poor_environment,no_control,multi_vehicle,vulnerable_x_dark,Initial_Impact_Type_Approaching,Initial_Impact_Type_Other,Initial_Impact_Type_Rear end,Initial_Impact_Type_SMV other,Initial_Impact_Type_SMV unattended vehicle,Initial_Impact_Type_Sideswipe,Initial_Impact_Type_Turning movement,Road_1_Surface_Condition_Ice,Road_1_Surface_Condition_Loose sand or gravel,Road_1_Surface_Condition_Loose snow,Road_1_Surface_Condition_Mud,Road_1_Surface_Condition_Other,Road_1_Surface_Condition_Packed snow,Road_1_Surface_Condition_Slush,Road_1_Surface_Condition_Spilled liquid,Road_1_Surface_Condition_Unknown,Road_1_Surface_Condition_Wet,Environment_Condition_1_Drifting Snow,"Environment_Condition_1_Fog, mist, smoke, dust",Environment_Condition_1_Freezing Rain,Environment_Condition_1_Other,Environment_Condition_1_Rain,Environment_Condition_1_Snow,Environment_Condition_1_Strong wind,Environment_Condition_1_Unknown,Light_Dawn,Light_Daylight,Light_Dusk,Light_Other,Light_Unknown,Traffic_Control_MPS,Traffic_Control_No control,Traffic_Control_Other,Traffic_Control_Ped. crossover,Traffic_Control_Police control,Traffic_Control_Roundabout,Traffic_Control_School bus,Traffic_Control_School guard,Traffic_Control_Stop sign,Traffic_Control_Traffic controller,Traffic_Control_Traffic gate,Traffic_Control_Traffic signal,Traffic_Control_Yield sign
67359,0,2,0,0,0,3,1,0,1,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
26171,0,2,0,0,0,12,0,0,0,0,0,0,1,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0


In [112]:
# Verify final dataset dimensions after all feature engineering steps
print(f"Final dataset: {df_f.shape[0]:,} records x {df_f.shape[1]} features")

Final dataset: 91,843 records x 60 features


---
## Export — Save Engineered Feature Set

The final dataset is exported for model training and evaluation.

In [113]:
# Export the engineered dataset for model training
df_f.to_csv('final_features.csv', index=False)

print("File Saved.")

File Saved.
